第 8 章：MLP 多层感知机

学习前向传播、反向传播、激活函数、梯度消失与梯度爆炸、初始化、正则化。

项目：房价预测、分类任务、训练曲线分析。

一、使用 MLP 实现波士顿房价预测 HousingData.csv

In [13]:
import copy
import warnings

import torch
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

# 去除警告
warnings.filterwarnings("ignore")

# 设置全局随机值
torch.manual_seed(23)

In [14]:
# 0. 确定设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("使用设备：", device)

使用设备： mps


In [15]:
# 1、定义多层感知机模型 ， （回归任务）
class HousingDataMLP(nn.Module):
    def __init__(self):
        super(HousingDataMLP, self).__init__()
        # 输入特征 13 个 -> 设置中间隐藏层为 4 个 -> 最终输出 1 个预测的结果
        self.network = nn.Sequential(
            # 输入
            nn.Linear(13, 8),
            nn.ReLU(),
            # 中间隐藏层
            nn.Linear(8, 4),
            nn.ReLU(),
            # 输出
            nn.Linear(4, 1),
        )

    def forward(self, input):
        return self.network(input)


In [16]:
# 2、定义数据集对象
class HousingDataDataset(Dataset):
    def __init__(self, data: pd.DataFrame, target: pd.Series):
        self.features = torch.from_numpy(data.to_numpy()).float()
        self.target = torch.from_numpy(target.to_numpy()).float()

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.target[index]


In [17]:
# 3、加载数据
df = pd.read_csv("../data/ml/HousingData.csv")

In [30]:
# 4、分割数据
# 划分训练集、验证集、测试集：8:1:1
# 将每一个数据集的数据索引分割出来，之后拿着这些索引来获取对应的行数据，即可完成对应数据的分割
train_indices, valid_indices, test_indices = random_split(
    range(len(df)),
    [406, 50, 50],
    generator=torch.Generator().manual_seed(23)
)

# 5、数据预处理：只使用训练集统计量，避免验证集和测试集信息泄漏
feature_columns = df.columns[:-1] # 取出列特征tag
train_df = df.iloc[train_indices.indices].copy() # 通过索引把训练、验证和测试数据复制出来， .copy()创建一个独立的数据副本
valid_df = df.iloc[valid_indices.indices].copy()
test_df = df.iloc[test_indices.indices].copy()

# # 将所有特征列统一转换为 float64， 直接重新构造 DataFrame，确保特征列全部是 float64
train_df = train_df.astype({col: "float64" for col in feature_columns})
valid_df = valid_df.astype({col: "float64" for col in feature_columns})
test_df = test_df.astype({col: "float64" for col in feature_columns})

train_feature_means = train_df.loc[:, feature_columns].mean() # 获取训练数据集特征列中每一列的平均值
train_feature_stds = train_df.loc[:, feature_columns].std().replace(0, 1) # 获取训练数据集特征列中每一列的标准差。.replace(0, 1)表示将标准差为0的特征置为1，防止计算标准差的过程出现除0异常

# 缺失值的填充和标准化
for split_df in (train_df, valid_df, test_df): # 构造一个元组来进行遍历， 对训练、验证和测试集分别进行缺失值填充
    split_df.loc[:, feature_columns] = split_df.loc[:, feature_columns].fillna(train_feature_means) # 填充缺失值
    split_df.loc[:, feature_columns] = (split_df.loc[:, feature_columns] - train_feature_means) / train_feature_stds # X_std = (每个样本X - 均值u) / 标准差σ

# 构造数据集 dataset
train_set = HousingDataDataset(train_df.iloc[:, :-1], train_df.iloc[:, -1])
valid_set = HousingDataDataset(valid_df.iloc[:, :-1], valid_df.iloc[:, -1])
test_set = HousingDataDataset(test_df.iloc[:, :-1], test_df.iloc[:, -1])

# 6、定义数据加载器 DataLoader
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=16, shuffle=False)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

print(len(train_set), len(valid_set), len(test_set))
print(train_set[0])
# print(iter(train_loader))


406 50 50
(tensor([-0.4518,  0.9391, -1.3004, -0.2716, -0.7079,  0.8032,  0.0843, -0.2958,
        -0.2959, -1.0977, -0.0233,  0.4306, -0.7182]), tensor(28.2000))


In [33]:
# 5、训练

# 声明模型
model = HousingDataMLP().to(device)
print(model)
print(model.named_parameters())

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

best_valid_loss = float("inf")
best_state = None

epochs = 800
for epoch in range(epochs):
    # 训练阶段
    model.train()

    train_loss_sum = 0.0
    train_samples = 0

    for (x, target) in train_loader:
        x = x.to(device)
        target = target.to(device).reshape(-1, 1) # 由于模型输出是 [batch, 1]，因此这里的形状要和模型输出一致
        # 清空梯度
        optimizer.zero_grad(set_to_none=True)
        # 前向传播
        output = model(x)
        # 计算损失
        loss = criterion(output, target)
        # 计算梯度(反向传播)
        loss.backward()
        # 更新参数
        optimizer.step()
        # 累计损失
        batch_size = x.size(0)
        train_samples += batch_size
        train_loss_sum += loss.item() * batch_size

    train_loss = train_loss_sum / train_samples

    # 验证阶段
    valid_loss_sum = 0.0
    valid_samples = 0
    with torch.no_grad():
        model.eval()
        for (x, target) in valid_loader:
            x = x.to(device)
            target = target.to(device).reshape(-1, 1)
            output = model(x)
            loss = criterion(output, target)
            batch_size = x.size(0)
            valid_loss_sum += loss.item() * batch_size
            valid_samples += batch_size

    valid_loss = valid_loss_sum / valid_samples

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        # 必须复制参数，不能直接保存 state_dict() 的引用
        best_state = copy.deepcopy(model.state_dict())
        # best_state = {
        #     k: v.detach().cpu().clone()
        #     for k, v in model.state_dict().items()
        # }

    if (epoch + 1) % 10 == 0:
        print(f"epoch: {epoch + 1}, train_loss: {train_loss:.4f}, valid_loss: {valid_loss:.4f}")


# 6、保存 最佳模型参数
torch.save(best_state, "HousingDataMLP_model.pth")
print("最佳模型已保存：HousingDataMLP_model.pth", "best_valid_loss: ", best_valid_loss)

HousingDataMLP(
  (network): Sequential(
    (0): Linear(in_features=13, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )
)
<generator object Module.named_parameters at 0x123107640>
epoch: 10, train_loss: 73.7373, valid_loss: 57.7949
epoch: 20, train_loss: 24.5167, valid_loss: 27.6505
epoch: 30, train_loss: 18.9662, valid_loss: 21.2555
epoch: 40, train_loss: 15.6001, valid_loss: 17.9882
epoch: 50, train_loss: 13.8598, valid_loss: 15.6086
epoch: 60, train_loss: 12.9955, valid_loss: 14.6186
epoch: 70, train_loss: 12.0333, valid_loss: 13.2681
epoch: 80, train_loss: 11.4632, valid_loss: 11.3575
epoch: 90, train_loss: 11.2167, valid_loss: 11.2615
epoch: 100, train_loss: 10.6932, valid_loss: 10.9796
epoch: 110, train_loss: 9.9796, valid_loss: 10.1096
epoch: 120, train_loss: 9.4949, valid_loss: 9.4942
epoch: 130, train_loss: 9.0189, valid_loss: 8.8661
epoch: 140, t

In [34]:
# 7、使用测试集进行模型测试
test_model = HousingDataMLP().to(device)
test_model.load_state_dict(torch.load("HousingDataMLP_model.pth", map_location=device, weights_only=True))
test_model.eval()

test_loss_sum = 0.0
test_samples = 0
with torch.inference_mode():
    for (x, target) in test_loader:
        x = x.to(device)
        target = target.to(device).reshape(-1, 1)
        output = test_model(x)
        loss = criterion(output, target)
        batch_size = x.size(0)
        test_loss_sum += loss.item() * batch_size
        test_samples += batch_size
        print(f"target: {target}, output: {output}, loss: {loss:.4f}")

test_loss = test_loss_sum / test_samples
print(f"test_loss: {test_loss:.4f}")


target: tensor([[27.5000],
        [15.4000],
        [13.4000],
        [14.1000],
        [30.5000],
        [18.9000],
        [44.0000],
        [13.6000],
        [17.0000],
        [23.2000],
        [18.0000],
        [ 8.3000],
        [26.4000],
        [22.2000],
        [24.1000],
        [22.0000]], device='mps:0'), output: tensor([[23.1253],
        [16.2382],
        [11.2197],
        [12.3180],
        [25.8081],
        [17.1067],
        [38.6237],
        [12.6595],
        [12.5873],
        [24.1595],
        [18.5118],
        [10.2342],
        [24.3181],
        [22.5840],
        [23.6477],
        [25.3958]], device='mps:0'), loss: 7.7126
target: tensor([[50.0000],
        [18.2000],
        [20.8000],
        [31.6000],
        [23.3000],
        [22.5000],
        [11.5000],
        [18.4000],
        [19.4000],
        [33.4000],
        [ 8.8000],
        [22.3000],
        [22.1000],
        [50.0000],
        [ 9.5000],
        [23.3000]], device='mps:0'